## Preprocess AMI into required format


In [11]:
import json
import glob
import os

def combine_sort_transcripts(folder_path: str) -> dict:
    """
    Load Closeup1–4 transcript files, combine them,
    sort by start_time, and return:
      { "1": "utterance", "2": "utterance", ... }
    """
    OUT_PATH = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/transcript"
    for file in os.listdir(folder_path):
        meeting_id = file.split(".")[0]
        out_path = os.path.join(OUT_PATH, f"{meeting_id}.json")
        all_utterances = []
        for path in glob.glob(os.path.join(folder_path, f"{meeting_id}.*.utterances.json")):
            print(f"Processing file: {path}")
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            for item in data.values():
                if "start_time" in item and "text" in item:
                    all_utterances.append(item)

    # Global sort by start_time
        all_utterances.sort(key=lambda x: float(x["start_time"]))

        # Re-index from 1, keep only text
        combined = {
            str(i + 1): u["text"].strip()
            for i, u in enumerate(all_utterances)
        }
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(combined, f, ensure_ascii=False, indent=2)




# ---------- usage ----------
# folder = "path/to/transcripts"
# combined = combine_sort_transcripts(folder)

# with open("combined_transcript.json", "w", encoding="utf-8") as f:
#     json.dump(combined, f, ensure_ascii=False, indent=2)


In [12]:
FILE_ROOT = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json"
combine_sort_transcripts(FILE_ROOT)


Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1007c.D.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1007c.C.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1007c.B.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1007c.A.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1004c.C.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1004c.D.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1004c.B.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/IS1004c.A.utterances.json


In [17]:
import json
import glob
import os
import re


def combine_sort_transcripts_with_speaker(folder_path: str):
    """
    Load Closeup1–4 transcript files, combine them,
    sort by start_time, and save:
      {
        "1": { "speaker": "speaker1", "text": "utterance" },
        "2": { "speaker": "speaker2", "text": "utterance" },
        ...
      }
    """

    OUT_PATH = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/transcript_json_filtered_emotion"
    os.makedirs(OUT_PATH, exist_ok=True)

    # Collect unique meeting IDs
    meeting_ids = set(f.split(".")[0] for f in os.listdir(folder_path))

    for meeting_id in meeting_ids:
        out_path = os.path.join(OUT_PATH, f"{meeting_id}.json")
        all_utterances = []

        for path in glob.glob(os.path.join(folder_path, f"{meeting_id}.*.utterances.json")):
            print(f"Processing file: {path}")

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            for item in data.values():
                    if item["speakers"] == "Closeup1":
                        speaker_name = "Speaker1"
                    elif item["speakers"] == "Closeup2":
                        speaker_name = "Speaker2"
                    elif item["speakers"] == "Closeup3":
                        speaker_name = "Speaker3"
                    elif item["speakers"] == "Closeup4":
                        speaker_name = "Speaker4"
                    print(item)
                    all_utterances.append({
                        "start_time": float(item["start_time"]),
                        "utterance": item["text"].strip(),
                        "speaker": speaker_name,
                        "emotion": item["emotion"]["emotion"] if item["emotion"]!=None else "neutral"
                    })

        # Sort globally by time
        all_utterances.sort(key=lambda x: x["start_time"])

        # Re-index from 1
        combined = {
            str(i + 1): {
                "speaker": u["speaker"],
                "utterance": u["utterance"],
                "emotion": u["emotion"]
            }
            for i, u in enumerate(all_utterances)
        }

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(combined, f, ensure_ascii=False, indent=2)

        print(f"✅ Saved combined transcript with speakers: {out_path}")

# ---------- usage ----------
FOLDER = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_emotion"
combine_sort_transcripts_with_speaker(FOLDER)


Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_emotion/IS1007c.D.utterances.json
{'text': 'Yeah.', 'start_time': 812.618, 'end_time': 813.367, 'speakers': 'Closeup4', 'emotion': {'emotion': 'neutral', 'confidence': None}}
{'text': 'Yeah.', 'start_time': 816.512, 'end_time': 817.024, 'speakers': 'Closeup4', 'emotion': {'emotion': 'happy', 'confidence': None}}
{'text': 'Oh.', 'start_time': 961.84, 'end_time': 962.192, 'speakers': 'Closeup4', 'emotion': {'emotion': 'neutral', 'confidence': None}}
{'text': "But people are often enough looking at the help, once they see the help button they say oh this is a complicated stuff.  It's a psychology.  Okay. And let us see what the market demands. We could just go to my presentation.", 'start_time': 968.998, 'end_time': 986.336, 'speakers': 'Closeup4', 'emotion': {'emotion': 'neutral', 'confidence': None}}
{'text': "Yeah that's right.", 'start_time': 989.952, 'end_time': 990.906, 'speakers': 'Closeup

In [18]:
import json
import glob
import os

def combine_sort_transcripts_to_txt(folder_path: str):
    """
    Load Closeup1–4 transcript files, combine them, sort by start_time,
    and save ONE .txt per meeting:
      line 1 = utterance 1
      line 2 = utterance 2
      ...
    """
    OUT_PATH = "/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/transcript_txt"
    os.makedirs(OUT_PATH, exist_ok=True)

    # Get unique meeting IDs from filenames in the folder
    meeting_ids = set(f.split(".")[0] for f in os.listdir(folder_path))

    for meeting_id in sorted(meeting_ids):
        all_utterances = []

        # Gather all closeup utterance files for this meeting
        pattern = os.path.join(folder_path, f"{meeting_id}.*.utterances.json")
        paths = glob.glob(pattern)

        if not paths:
            continue

        for path in paths:
            print(f"Processing file: {path}")
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            for item in data.values():
                if "start_time" in item and "text" in item:
                    all_utterances.append({
                        "start_time": float(item["start_time"]),
                        "text": item["text"]
                    })

        # Sort globally by start_time
        all_utterances.sort(key=lambda x: x["start_time"])

        # Write to .txt (one utterance per line)
        out_path = os.path.join(OUT_PATH, f"{meeting_id}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            for u in all_utterances:
                line = u["text"].strip()
                if line:
                    f.write(line + "\n")

        print(f"✅ Saved: {out_path}")


In [19]:
combine_sort_transcripts_to_txt("/Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json")

Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2002a.A.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2002a.B.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2002a.D.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2002a.C.utterances.json
✅ Saved: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/transcript_txt/ES2002a.txt
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2003a.C.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2003a.D.utterances.json
Processing file: /Users/linyunxiang/NL/TUD/Project/2025/dynamic_summary/dataset/ami/utterance_json/ES2003a.B.utterances.json
Processing file: /User